In [ ]:
!pip install transformers datasets torch scikit-learn -q

import torch
print("GPU Available:", torch.cuda.is_available())

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

print("Dataset loaded!")
print(dataset)
print("\nExample:")
print(dataset['train'][0])
print("\nLabel names:", dataset['train'].features['label'].names)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_data = dataset['train'].shuffle(seed=42).select(range(3000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_tokenized = train_data.map(tokenize, batched=True)
test_tokenized = test_data.map(tokenize, batched=True)

train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization complete!")
print(f"Training examples: {len(train_tokenized)}")
print(f"Testing examples: {len(test_tokenized)}")

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4
)

print("BERT model loaded!")
print("Labels: World, Sports, Business, Sci/Tech")

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./bert_news_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics
)

print("Training started — please wait 10-15 minutes...")
trainer.train()
print("Training complete!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

predictions = trainer.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

label_names = ['World', 'Sports', 'Business', 'Sci/Tech']

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names,
            yticklabels=label_names)
plt.title('BERT News Classifier — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(f"\nFinal Accuracy: {accuracy_score(labels, preds):.4f}")
print(f"Final F1 Score: {f1_score(labels, preds, average='weighted'):.4f}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

label_names = ['World', 'Sports', 'Business', 'Sci/Tech']

def predict_news(headline):
    inputs = tokenizer(
        headline,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    confidence = torch.softmax(outputs.logits, dim=1).max().item()

    return f"Category: {label_names[pred]} (Confidence: {confidence:.2%})"

headlines = [
    "Apple launches new iPhone with AI features",
    "Manchester United wins championship title",
    "Stock market crashes amid economic fears",
    "US and China sign new trade agreement",
    "Pakistan wins cricket world cup",
    "NASA discovers water on Mars"
]

print("Live Predictions:")
print("=" * 60)
for h in headlines:
    print(f"Headline: {h}")
    print(f"Result: {predict_news(h)}")
    print("-" * 60)

In [ ]:
model.save_pretrained('./bert_news_classifier')
tokenizer.save_pretrained('./bert_news_classifier')
print("Model saved successfully!")